<a href="https://www.kaggle.com/code/adityabayhaqie/qwen2-7b-fine-tune-using-qlora-nlaw?scriptVersionId=308713443" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Install Dependencies

In [1]:
%%capture
# 1. Force un-install potentially conflicting libraries first
!pip uninstall -y unsloth unsloth-zoo peft trl transformers

# 2. Install Unsloth and compatible dependencies
# We use the specific 'colab-new' tag which is stable for T4 environments like Kaggle/Colab
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install other requirements without deps to prevent version overwrites
!pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes unsloth-zoo

## Import Libraries and Setup

In [2]:
import torch
from unsloth import FastLanguageModel
import json
import pandas as pd
from datasets import Dataset
import os

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

print(f"GPU Model: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-03 16:00:53.033010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775232053.417577      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775232053.528374      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775232054.235952      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775232054.235993      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775232054.235996      24 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Model: Tesla T4


## Load, Merge, and Count Data

In [3]:
train_file_paths = [
    "/kaggle/input/nusantara-law-corpus/Adagium/adagium-all.json",
    "/kaggle/input/nusantara-law-corpus/GBHN/GBHN-all.json",
    "/kaggle/input/nusantara-law-corpus/Glosarium-MA/GMA-all.json",
    "/kaggle/input/nusantara-law-corpus/HukumOnline/HO-all.json",
    "/kaggle/input/nusantara-law-corpus/KHPTSultra/KHPTS-all.json",
    "/kaggle/input/nusantara-law-corpus/LawDictionary/LD-all.json",
    "/kaggle/input/nusantara-law-corpus/TAP-MPR/TMPR-all.json",
    "/kaggle/input/nusantara-law-corpus/UUD/uud-id.json"
]

test_file_path = "/kaggle/input/nusantara-law-corpus/test-data.json"

combined_train_data = []

for file_path in train_file_paths:
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_train_data.extend(data)
                    print(f"Successfully loaded {len(data)} records from: {os.path.basename(file_path)}")
                else:
                    print(f"Warning: {file_path} format is not a list of records.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

test_data = []
if os.path.exists(test_file_path):
    try:
        with open(test_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                test_data.extend(data)
                print(f"Successfully loaded {len(data)} test records from: {os.path.basename(test_file_path)}")
    except Exception as e:
        print(f"Error reading {test_file_path}: {e}")
else:
    print(f"Test file not found: {test_file_path}")

train_df = pd.DataFrame(combined_train_data)
test_df = pd.DataFrame(test_data)

print(f"Total train data points: {len(train_df)}")
print(f"Total test data points: {len(test_df)}")

raw_train_dataset = Dataset.from_pandas(train_df)
raw_eval_dataset = Dataset.from_pandas(test_df)

Successfully loaded 89 records from: adagium-all.json
Successfully loaded 33 records from: GBHN-all.json
Successfully loaded 207 records from: GMA-all.json
Successfully loaded 2342 records from: HO-all.json
Successfully loaded 144 records from: KHPTS-all.json
Successfully loaded 2456 records from: LD-all.json
Successfully loaded 352 records from: TMPR-all.json
Successfully loaded 250 records from: uud-id.json
Successfully loaded 444 test records from: test-data.json
Total train data points: 5873
Total test data points: 444


## Load Model (Qwen2 7b)

In [4]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2-7b-bnb-4bit"
max_seq_length = 2048
dtype = None 
load_in_4bit = True 

print(f"Loading Model: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token 

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    responses    = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        text = alpaca_prompt.format(instruction, context, response) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

train_dataset = raw_train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = raw_eval_dataset.map(formatting_prompts_func, batched = True)

print(f"Success! Loaded {model_name} and formatted both train and test data.")

Loading Model: unsloth/Qwen2-7b-bnb-4bit...
==((====))==  Unsloth 2026.4.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5873 [00:00<?, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

Success! Loaded unsloth/Qwen2-7b-bnb-4bit and formatted both train and test data.


## Configure QLoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## Training (SFTTrainer)

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

sft_config = SFTConfig(
    output_dir="outputs",
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=6, 
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=175,        
    save_steps=350,        
    logging_steps=10,
    save_total_limit=1,          
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss", 
    greater_is_better=False,
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,   
    args = sft_config,
)

def predict_training_time(dataset, config, seconds_per_step=1.2):
    total_examples = len(dataset)
    batch_size = config.per_device_train_batch_size
    grad_accum = config.gradient_accumulation_steps
    epochs = config.num_train_epochs
    
    steps_per_epoch = total_examples // (batch_size * grad_accum)
    total_steps = steps_per_epoch * epochs
    
    estimated_seconds = total_steps * seconds_per_step
    hours = estimated_seconds // 3600
    minutes = (estimated_seconds % 3600) // 60
    
    print("Prediksi Estimasi Waktu Pelatihan")
    print(f"Total Data Latih: {total_examples} sampel")
    print(f"Total Steps: {total_steps}")
    print(f"Estimasi Waktu: ~{int(hours)} jam dan {int(minutes)} menit")
    print(f"(Berdasarkan asumsi kecepatan ~{seconds_per_step} detik per step pada Tesla T4)")

predict_training_time(train_dataset, sft_config)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5873 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Prediksi Estimasi Waktu Pelatihan
Total Data Latih: 5873 sampel
Total Steps: 4404
Estimasi Waktu: ~1 jam dan 28 menit
(Berdasarkan asumsi kecepatan ~1.2 detik per step pada Tesla T4)


## Execute Training

In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,873 | Num Epochs = 6 | Total steps = 4,410
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
175,1.045800,1.421783
350,0.985200,1.364480
525,0.880000,1.359034
700,0.829900,1.339192
875,0.734900,1.344811
1050,0.764600,1.359416
1225,0.892700,1.332708
1400,0.762700,1.331133
1575,0.760000,1.362628
1750,0.781500,1.363181


## Inference and Evaluation

In [8]:
import time
import random

FastLanguageModel.for_inference(model)

sample_indices = random.sample(range(len(eval_dataset)), min(5, len(eval_dataset)))

def generate_response(prompt, context=""):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                prompt,
                context,
                "", 
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    start_time = time.time()
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,       
        use_cache=True,
        temperature=0.6,          
        top_k=50,                 
        top_p=0.9,                
        repetition_penalty=1.1,   
        do_sample=True            
    )
    
    end_time = time.time()
    
    decoded_output = tokenizer.batch_decode(outputs)[0]
    response = decoded_output.split("### Response:\n")[-1].replace(tokenizer.eos_token, "")
    
    num_tokens = len(outputs[0])
    duration = end_time - start_time
    tokens_per_sec = num_tokens / duration
    
    return response, tokens_per_sec

print("Memulai Evaluasi menggunakan Test Dataset")

for i, idx in enumerate(sample_indices):
    sample = eval_dataset[idx]
    test_instruction = sample["instruction"]
    test_context = sample.get("context", "")
    
    print(f"Test Case {i+1}: {test_instruction}")
    response, speed = generate_response(test_instruction, test_context)
    print(f"Response:\n{response}")
    print(f"Speed: {speed:.2f} tokens/sec\n")

Memulai Evaluasi menggunakan Test Dataset
Test Case 1: Apa yang dimaksud dengan Banding dalam konteks hukum?
Response:
Banding adalah upaya hukum yang dilakukan oleh Pihak Pelaku Perjanjian atau pihak lain yang mengalami kerugian berdasarkan putusan pengadilan atas permohonan Banding kepada Pengadilan Tinggi, Pengadilan Agung atau Mahkamah Agung.
Speed: 31.11 tokens/sec

Test Case 2: Jelaskan definisi dari istilah hukum Barang bukti/corpus delicti.
Response:
Barang bukti adalah barang atau benda lain yang berhubungan dengan suatu perkara yang sedang diproses dan merupakan bagian dari korban atau pelaksana kejahatan yang diperiksa oleh penyidik untuk menentukan apakah terdapat kejahatan dan menemukan siapa pelakunya.
Speed: 30.76 tokens/sec

Test Case 3: Apa definisi BUMD (Badan Usaha Milik Daerah)?
Response:
BUMD adalah badan usaha yang seluruh atau sebagian besar nilai sahamnya dimiliki oleh Pemerintah Daerah atau adanya kewajiban pembayaran hak kepentingan negara.
Speed: 39.77 tokens

## Cleanup Cell

In [9]:
import shutil
import os
import gc
import torch

# Hapus variabel yang tidak terpakai dari RAM
del trainer
gc.collect()

# Bersihkan Cache GPU
torch.cuda.empty_cache()

# Hapus folder checkpoint pelatihan
path_to_clean = "/kaggle/working/outputs"
if os.path.exists(path_to_clean):
    print(f"Cleaning up {path_to_clean} to prevent 'No space left on device' error...")
    try:
        shutil.rmtree(path_to_clean)
        print("Cleanup successful. Disk space reclaimed.")
    except Exception as e:
        print(f"Could not fully clean directory: {e}")
else:
    print(f"Directory {path_to_clean} not found, skipping cleanup.")

!df -h /kaggle/working

Cleaning up /kaggle/working/outputs to prevent 'No space left on device' error...
Cleanup successful. Disk space reclaimed.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15M   20G   1% /kaggle/working


## Save the Model

In [10]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(hf_token)

repo_name = "bayhaqieee/qwen2-7b-nlaw-gguf" 

# PUSH ADAPTERS
print("Pushing Adapters (LoRA) to Hugging Face...")
try:
    model.push_to_hub(repo_name, token=hf_token)
    tokenizer.push_to_hub(repo_name, token=hf_token)
    print("Adapters (LoRA) Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"Adapter Push Failed: {e}")

# PUSH GGUF KE HUGGING FACE
print("\nPushing GGUF to Hugging Face (This requires heavy disk space)...")
try:
    model.push_to_hub_gguf(
        repo_name, 
        tokenizer, 
        quantization_method = "q4_k_m",
        token = hf_token
    )
    print("GGUF Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"\nGGUF Push Failed: {e}")
    print("\nNOTE: Kaggle's 20GB disk limit often blocks 7B GGUF conversions.")
    print("TAPI JANGAN KHAWATIR! Adapter LoRA Anda sudah berhasil disimpan ke Hugging Face di Langkah 1!")
    print("Anda bisa menggabungkan (merge) LoRA ke Base Model menjadi GGUF secara terpisah di Google Colab.")

Pushing Adapters (LoRA) to Hugging Face...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/bayhaqieee/qwen2-7b-nlaw-gguf


...

[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_hi9ke7ik_gguf/qwen2-7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_hi9ke7ik_gguf/qwen2-7b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/qwen2-7b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_hi9ke7ik_gguf/qwen2-7b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading qwen2-7b.Q4_K_M.gguf...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/bayhaqieee/qwen2-7b-nlaw-gguf
Unsloth: Cleaning up temporary files...
GGUF Pushed to Hugging Face successfully!


In [11]:
# print("\nSaving Adapters Locally (Kaggle)")
# local_folder = "qwen2-7b-nlaw_adapter"
# model.save_pretrained(local_folder)
# tokenizer.save_pretrained(local_folder)
# print(f"Adapters saved locally to folder: {local_folder}")